# KONE Oyj — Financial Analysis
### Chau Minh Tran | Junior Financial Management | Aalto University
---
**Source:** KONE Annual Review 2025 (KNEBV: Nasdaq Helsinki)  
**Analysis covers:** Revenue & growth trends | Cost structure | Margin decomposition | Budget forecasting | P&L variance  
**Tools:** Python · Pandas · Plotly · NumPy

## 0. Setup & Data Ingestion

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Plotly theme ──
BLUE      = '#1a4fd6'
BLUE_LIGHT= '#3b6ef5'
GREEN     = '#15803d'
AMBER     = '#b45309'
RED       = '#dc2626'
GRAY      = '#94a3b8'
BG        = '#f4f6fb'
PANEL     = '#ffffff'

LAYOUT = dict(
    plot_bgcolor=PANEL,
    paper_bgcolor=BG,
    font=dict(family='Inter, sans-serif', color='#1e2a3a', size=12),
    title_font=dict(family='Inter, sans-serif', size=16, color='#1e2a3a'),
    legend=dict(bgcolor='rgba(0,0,0,0)', bordercolor='rgba(26,79,214,0.2)', borderwidth=1),
    margin=dict(t=60, b=50, l=60, r=40),
    xaxis=dict(gridcolor='rgba(26,79,214,0.08)', linecolor='rgba(26,79,214,0.2)'),
    yaxis=dict(gridcolor='rgba(26,79,214,0.08)', linecolor='rgba(26,79,214,0.2)'),
)
print("✅ Libraries loaded successfully")

## 1. Real KONE Financial Data
Data sourced directly from **KONE Annual Review 2025** — consolidated financial statements and key figures table.

In [ ]:
# ── Annual income statement (MEUR) ──
# Source: KONE Annual Review 2025, Key figures and financial development (p.30-31)
income_data = {
    'Year':         [2021,   2022,    2023,    2024,    2025],
    'Sales':        [10514,  10907,   10952,   11098,   11245],
    'EBIT':         [1295,   1031,    1200,    1249,    1336],
    'Adj_EBIT':     [1310,   1077,    1248,    1303,    1369],
    'Net_Income':   [1023,   784,     932,     961,     992],
    'Income_Before_Tax': [1321, 1028, 1206,   1254,    1327],
    'Cash_Flow_Ops':[1829,   755,     1485,    1589,    1761],
    'CapEx':        [217,    209,     322,     397,     378],
    'R_and_D':      [189,    188,     185,     204,     234],
    'Orders':       [8853,   9131,    8578,    8759,    9087],
    'Order_Book':   [8564,   9026,    8716,    9059,    8804],
    'Employees':    [61698,  63186,   63164,   64072,   64294],
}
df_income = pd.DataFrame(income_data)

# Derived margin metrics
df_income['EBIT_Margin_%']     = (df_income['EBIT'] / df_income['Sales'] * 100).round(1)
df_income['Adj_EBIT_Margin_%'] = (df_income['Adj_EBIT'] / df_income['Sales'] * 100).round(1)
df_income['Net_Margin_%']      = (df_income['Net_Income'] / df_income['Sales'] * 100).round(1)
df_income['Tax_Rate_%']        = ((1 - df_income['Net_Income'] / df_income['Income_Before_Tax']) * 100).round(1)
df_income['YoY_Sales_%']       = df_income['Sales'].pct_change() * 100
df_income['RD_as_%_Sales']     = (df_income['R_and_D'] / df_income['Sales'] * 100).round(2)
df_income['CapEx_as_%_Sales']  = (df_income['CapEx'] / df_income['Sales'] * 100).round(2)
df_income['FCF']               = df_income['Cash_Flow_Ops'] - df_income['CapEx']

# ── FY2025 cost breakdown (MEUR) ──
# Source: KONE Annual Review 2025, Note 2.2 Costs and Expenses (p.98)
costs_2025 = {
    'Category': ['Direct Materials & Subcontracting', 'Wages & Employment Costs',
                 'Other Production Costs', 'SG&A', 'Depreciation & Amortisation', 'R&D'],
    'FY2025_MEUR': [3877.9, 4090.7, 879.2, 782.8, 319.9, 233.9],
    'FY2024_MEUR': [3947.5, 3907.0, 882.4, 860.0, 292.2, 203.6],
}
df_costs = pd.DataFrame(costs_2025)
df_costs['YoY_Change_%'] = ((df_costs['FY2025_MEUR'] - df_costs['FY2024_MEUR']) / df_costs['FY2024_MEUR'] * 100).round(1)
df_costs['%_of_Sales_2025'] = (df_costs['FY2025_MEUR'] / 11245 * 100).round(1)

# ── Revenue by business line FY2024 vs FY2025 ──
# Source: KONE Annual Review 2025, Note 2.1 Sales (p.97)
df_segment = pd.DataFrame({
    'Segment': ['New Building Solutions', 'Service (Maintenance)', 'Modernisation'],
    'FY2024_MEUR': [4506.9, 4503.6, 2088.0],
    'FY2025_MEUR': [4097.7, 4753.6, 2394.0],
    'FY2024_%': [41, 41, 19],
    'FY2025_%': [36, 42, 21],
})
df_segment['YoY_Change_%'] = ((df_segment['FY2025_MEUR'] - df_segment['FY2024_MEUR']) / df_segment['FY2024_MEUR'] * 100).round(1)

# ── Revenue by geography FY2024 vs FY2025 ──
df_geo = pd.DataFrame({
    'Region': ['Europe', 'Americas', 'Greater China', 'APMEA'],
    'FY2024_MEUR': [4233.8, 2727.1, 2528.2, 1609.3],
    'FY2025_MEUR': [4524.4, 2812.1, 2166.0, 1742.7],
    'FY2024_%': [38, 25, 23, 14],
    'FY2025_%': [40, 25, 19, 15],
})
df_geo['YoY_Change_%'] = ((df_geo['FY2025_MEUR'] - df_geo['FY2024_MEUR']) / df_geo['FY2024_MEUR'] * 100).round(1)

# ── Balance sheet highlights ──
df_balance = pd.DataFrame({
    'Year':           [2021,    2022,    2023,    2024,    2025],
    'Total_Assets':   [9720,    9090,    8731,    9284,    9052],
    'Total_Equity':   [2867,    2786,    2893,    2827,    3199],
    'Net_Debt':       [-2164,   -1309,   -1013,   -831,    -700],
    'ROE_%':          [32.0,    25.9,    33.0,    33.8,    34.7],
    'ROCE_%':         [26.8,    22.4,    27.8,    27.2,    26.9],
    'Equity_Ratio_%': [41.2,    40.3,    40.9,    39.8,    39.9],
})

print("✅ KONE financial data loaded")
print(f"\n📊 Income statement: {len(df_income)} years | Costs: {len(df_costs)} categories")
print(f"   Segments: {len(df_segment)} | Geographies: {len(df_geo)}")
print("\n--- Income Statement Summary ---")
print(df_income[['Year','Sales','EBIT','EBIT_Margin_%','Net_Income','YoY_Sales_%']].to_string(index=False))

## 2. Revenue & Growth Trend Analysis
10-year KONE revenue performance with EBIT margin overlay, YoY growth rates, and segment/geographic breakdown.

In [ ]:
# ── Chart 1: Revenue trend + EBIT margin overlay ──
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(
    x=df_income['Year'], y=df_income['Sales'],
    name='Revenue (MEUR)', marker_color=BLUE_LIGHT,
    marker_line_color=BLUE, marker_line_width=1.5,
    opacity=0.85, text=df_income['Sales'].apply(lambda x: f'{x:,}'),
    textposition='outside', textfont=dict(size=10, color='#1e2a3a'),
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_income['Year'], y=df_income['EBIT_Margin_%'],
    name='EBIT Margin %', mode='lines+markers',
    line=dict(color=GREEN, width=2.5),
    marker=dict(size=7, color=GREEN),
), secondary_y=True)

fig.add_trace(go.Scatter(
    x=df_income['Year'], y=df_income['Adj_EBIT_Margin_%'],
    name='Adj. EBIT Margin %', mode='lines+markers',
    line=dict(color=AMBER, width=2, dash='dot'),
    marker=dict(size=5, color=AMBER),
), secondary_y=True)

fig.update_layout(
    title='KONE Revenue & EBIT Margin — FY2021 to FY2025',
    xaxis=dict(tickmode='array', tickvals=[2021,2022,2023,2024,2025], gridcolor='rgba(26,79,214,0.08)'),
    yaxis=dict(title='Revenue (MEUR)', gridcolor='rgba(26,79,214,0.08)', tickformat=','),
    yaxis2=dict(title='EBIT Margin %', range=[8, 15]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
    height=450
)
fig.show()
print(f"\n📈 FY2025 Revenue: €{df_income.iloc[-1]['Sales']:,}M | EBIT Margin: {df_income.iloc[-1]['EBIT_Margin_%']}%")
print(f"   5-Year CAGR: {((df_income.iloc[-1]['Sales']/df_income.iloc[0]['Sales'])**(1/4)-1)*100:.1f}%")

In [ ]:
# ── Chart 2: YoY Revenue Growth ──
yoy = df_income.dropna(subset=['YoY_Sales_%']).copy()
colors = [GREEN if x >= 0 else RED for x in yoy['YoY_Sales_%']]

fig = go.Figure(go.Bar(
    x=yoy['Year'], y=yoy['YoY_Sales_%'].round(1),
    marker_color=colors, marker_line_width=0,
    text=yoy['YoY_Sales_%'].round(1).apply(lambda x: f'{x:+.1f}%'),
    textposition='outside', textfont=dict(size=11),
))
fig.add_hline(y=0, line_color='#1e2a3a', line_width=1)
fig.update_layout(
    title='KONE Year-on-Year Revenue Growth (%)',
    yaxis=dict(title='YoY Growth %', ticksuffix='%', gridcolor='rgba(26,79,214,0.08)'),
    xaxis=dict(tickmode='array', tickvals=list(yoy['Year']), gridcolor='rgba(26,79,214,0.08)'),
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
    height=380,
)
fig.show()
print("\n📊 Key growth events:")
print("   FY2022: +3.7% — Ukraine/Russia impact, China slowdown begins")
print("   FY2023: +0.4% — Continued headwinds in new equipment markets")
print("   FY2024: +1.3% — Gradual recovery, service segment resilient")
print("   FY2025: +1.3% — Modernisation growth +14.7% drives recovery")

In [ ]:
# ── Chart 3: Revenue by Segment & Geography ──
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Revenue by Business Line (FY2025)', 'Revenue by Geography (FY2025)'],
    specs=[[{'type':'domain'},{'type':'domain'}]]
)

# Segment donut
fig.add_trace(go.Pie(
    labels=df_segment['Segment'], values=df_segment['FY2025_MEUR'],
    hole=0.55, name='Segment',
    marker=dict(colors=[BLUE, BLUE_LIGHT, '#93b8ff'],
                line=dict(color='white', width=2)),
    textinfo='label+percent', textfont_size=11,
), row=1, col=1)

# Geography donut
fig.add_trace(go.Pie(
    labels=df_geo['Region'], values=df_geo['FY2025_MEUR'],
    hole=0.55, name='Geography',
    marker=dict(colors=[BLUE, BLUE_LIGHT, '#93b8ff', '#c5d8ff'],
                line=dict(color='white', width=2)),
    textinfo='label+percent', textfont_size=11,
), row=1, col=2)

fig.update_layout(
    title='KONE Revenue Mix — FY2025',
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
    height=420,
)
fig.show()
print("\n🔑 Key insight: Service revenue (42%) overtook New Building Solutions (36%) in FY2025")
print("   Greater China declined from 23% → 19% of sales — structural headwind confirmed")

In [ ]:
# ── Chart 4: Segment Revenue YoY comparison ──
fig = go.Figure()
x = df_segment['Segment']

fig.add_trace(go.Bar(
    name='FY2024', x=x, y=df_segment['FY2024_MEUR'],
    marker_color='rgba(26,79,214,0.4)', marker_line_color=BLUE, marker_line_width=1,
    text=df_segment['FY2024_MEUR'].apply(lambda v: f'€{v:,.0f}M'),
    textposition='outside', textfont_size=10,
))
fig.add_trace(go.Bar(
    name='FY2025', x=x, y=df_segment['FY2025_MEUR'],
    marker_color=BLUE, marker_line_color=BLUE, marker_line_width=1,
    text=df_segment['FY2025_%'].apply(lambda v: f'{v}% of sales'),
    textposition='outside', textfont_size=10,
))
fig.add_trace(go.Scatter(
    name='YoY Change', x=x, y=df_segment['YoY_Change_%'],
    mode='markers+text', yaxis='y2',
    marker=dict(size=10, color=[GREEN if v>=0 else RED for v in df_segment['YoY_Change_%']], symbol='diamond'),
    text=df_segment['YoY_Change_%'].apply(lambda v: f'{v:+.1f}%'),
    textposition='top center', textfont=dict(size=10, color='#1e2a3a'),
))
fig.update_layout(
    title='Revenue by Business Line — FY2024 vs FY2025',
    barmode='group',
    yaxis=dict(title='Revenue (MEUR)', tickformat=',', gridcolor='rgba(26,79,214,0.08)'),
    yaxis2=dict(title='YoY Change %', overlaying='y', side='right', ticksuffix='%', showgrid=False),
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
    height=450,
)
fig.show()

## 3. Cost Structure & Margin Decomposition
Detailed breakdown of KONE's cost base from the FY2025 Annual Report, with margin bridge and efficiency trends.

In [ ]:
# ── Chart 5: Cost waterfall FY2025 ──
categories = df_costs['Category'].tolist() + ['EBIT']
values_2025 = df_costs['FY2025_MEUR'].tolist()
ebit = 11245 - sum(values_2025[:5])  # Simplified (excl R&D already in wages)
true_ebit = 1336

# Waterfall: start with revenue, subtract each cost
base = [0] + list(np.cumsum(values_2025[:-1]))  # running base (R&D embedded in wages)

fig = go.Figure(go.Waterfall(
    name='Cost Bridge',
    orientation='v',
    measure=['absolute','relative','relative','relative','relative','relative','total'],
    x=['Revenue','Materials &\nSubcontracting','Wages &\nEmployment','Other\nProduction','SG&A','D&A','EBIT'],
    y=[11245, -3877.9, -4090.7, -879.2, -782.8, -319.9, 0],
    connector=dict(line=dict(color='rgba(26,79,214,0.3)', width=1)),
    decreasing=dict(marker=dict(color='rgba(220,38,38,0.7)')),
    increasing=dict(marker=dict(color='rgba(21,128,61,0.7)')),
    totals=dict(marker=dict(color=BLUE)),
    text=['€11,245M','-€3,878M','-€4,091M','-€879M','-€783M','-€320M','€1,295M'],
    textposition='outside',
))
fig.update_layout(
    title='KONE Revenue to EBIT Waterfall — FY2025 (MEUR)',
    yaxis=dict(title='MEUR', tickformat=',', gridcolor='rgba(26,79,214,0.08)'),
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
    height=480,
)
fig.show()
print(f"\n💡 EBIT Margin FY2025: {true_ebit/11245*100:.1f}% — recovering from 9.5% trough in FY2022")

In [ ]:
# ── Chart 6: Cost categories as % of sales with YoY comparison ──
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Cost as % of Sales — FY2024 vs FY2025', 'YoY Cost Change (%)'])

# Horizontal bars: % of sales
fig.add_trace(go.Bar(
    name='FY2024', y=df_costs['Category'],
    x=(df_costs['FY2024_MEUR'] / 11098 * 100).round(1),
    orientation='h', marker_color='rgba(26,79,214,0.4)',
    text=(df_costs['FY2024_MEUR'] / 11098 * 100).round(1).apply(lambda v: f'{v:.1f}%'),
    textposition='auto',
), row=1, col=1)
fig.add_trace(go.Bar(
    name='FY2025', y=df_costs['Category'],
    x=df_costs['%_of_Sales_2025'],
    orientation='h', marker_color=BLUE,
    text=df_costs['%_of_Sales_2025'].apply(lambda v: f'{v:.1f}%'),
    textposition='auto',
), row=1, col=1)

# YoY change
colors = [GREEN if v <= 0 else RED for v in df_costs['YoY_Change_%']]
fig.add_trace(go.Bar(
    name='YoY Δ', y=df_costs['Category'],
    x=df_costs['YoY_Change_%'],
    orientation='h',
    marker_color=colors,
    text=df_costs['YoY_Change_%'].apply(lambda v: f'{v:+.1f}%'),
    textposition='outside',
), row=1, col=2)
fig.add_vline(x=0, line_color='#1e2a3a', line_width=1, row=1, col=2)

fig.update_layout(
    title='KONE Cost Structure Analysis — FY2024 vs FY2025',
    barmode='group', height=400,
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
)
fig.show()
print("\n📊 FY2025 Cost highlights:")
print(f"   Wages grew +4.7% YoY → largest absolute cost (€4,091M, 36.4% of sales)")
print(f"   Materials declined -1.8% → supply chain normalisation")
print(f"   SG&A improved -8.9% YoY → includes items affecting comparability (restructuring)")

In [ ]:
# ── Chart 7: Margin trends ──
fig = go.Figure()

for col, name, color, dash in [
    ('EBIT_Margin_%', 'EBIT Margin', BLUE, 'solid'),
    ('Adj_EBIT_Margin_%', 'Adj. EBIT Margin', GREEN, 'dot'),
    ('Net_Margin_%', 'Net Margin', AMBER, 'dash'),
]:
    fig.add_trace(go.Scatter(
        x=df_income['Year'], y=df_income[col],
        name=name, mode='lines+markers',
        line=dict(color=color, width=2.5, dash=dash),
        marker=dict(size=8, color=color),
        text=df_income[col].apply(lambda v: f'{v:.1f}%'),
        textposition='top center', textfont=dict(size=9),
    ))

# Add target reference line
fig.add_hline(y=12.5, line_color=RED, line_dash='dot', line_width=1.5,
              annotation_text='Strategy target ~12.5%', annotation_position='bottom right')

fig.update_layout(
    title='KONE Profitability Margins — FY2021 to FY2025',
    yaxis=dict(title='Margin %', ticksuffix='%', range=[7,15], gridcolor='rgba(26,79,214,0.08)'),
    xaxis=dict(tickmode='array', tickvals=[2021,2022,2023,2024,2025], gridcolor='rgba(26,79,214,0.08)'),
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
    height=420,
)
fig.show()

## 4. Budget vs Actuals Variance Analysis
Comparing KONE's actual FY2025 results against consensus analyst estimates (Bloomberg consensus), highlighting where performance beat or missed expectations.

In [ ]:
# ── Budget vs Actuals: FY2025 consensus vs actual ──
# Consensus estimates sourced from analyst coverage pre-results
variance_data = {
    'Metric':    ['Sales', 'EBIT', 'Adj. EBIT', 'Net Income', 'EPS (€)', 'Cash Flow from Ops', 'Orders Received'],
    'Consensus': [11180,   1310,   1355,         975,          1.88,      1620,                  8900],
    'Actual':    [11245,   1336,   1369,         992,          1.89,      1761,                  9087],
    'Unit':      ['MEUR',  'MEUR', 'MEUR',       'MEUR',       '€',       'MEUR',                'MEUR'],
}
df_var = pd.DataFrame(variance_data)
df_var['Variance']   = df_var['Actual'] - df_var['Consensus']
df_var['Variance_%'] = ((df_var['Variance'] / df_var['Consensus']) * 100).round(2)
df_var['Beat?']      = df_var['Variance'] >= 0

print("=" * 72)
print(f"{'Metric':<25} {'Consensus':>10} {'Actual':>10} {'Variance':>10} {'Var %':>8} {'Flag':>6}")
print("-" * 72)
for _, r in df_var.iterrows():
    flag = '✅ Beat' if r['Beat?'] else '❌ Miss'
    print(f"{r['Metric']:<25} {r['Consensus']:>10,.0f} {r['Actual']:>10,.0f} {r['Variance']:>+10,.0f} {r['Variance_%']:>7.1f}% {flag}")
print("=" * 72)

In [ ]:
# ── Chart 8: Variance waterfall ──
colors = [GREEN if v >= 0 else RED for v in df_var['Variance_%']]

fig = go.Figure(go.Bar(
    x=df_var['Metric'], y=df_var['Variance_%'],
    marker_color=colors, marker_line_width=0,
    text=df_var['Variance_%'].apply(lambda v: f'{v:+.2f}%'),
    textposition='outside', textfont=dict(size=11),
))
fig.add_hline(y=0, line_color='#1e2a3a', line_width=1.5)
fig.update_layout(
    title='FY2025 Actuals vs Consensus Estimates — Variance (%)',
    yaxis=dict(title='Variance vs Consensus (%)', ticksuffix='%', gridcolor='rgba(26,79,214,0.08)',
               zeroline=True, zerolinecolor='#1e2a3a', zerolinewidth=1.5),
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
    height=420,
)
fig.show()
print("\n✅ KONE beat consensus on all key metrics in FY2025")
print("   Biggest outperformance: Cash Flow from Operations (+8.7% vs consensus)")

## 5. Time Series Forecasting
Simple revenue forecast using linear trend + KONE management guidance (FY2026 guidance: mid-single-digit growth). Three scenarios modelled: Base, Bull, Bear.

In [ ]:
import numpy as np
from scipy import stats

# ── Fit linear trend on historical sales ──
years = df_income['Year'].values
sales = df_income['Sales'].values

slope, intercept, r_value, p_value, std_err = stats.linregress(years, sales)
print(f"Linear trend: slope = {slope:.0f} MEUR/year, R² = {r_value**2:.3f}")

# Forecast years
forecast_years = np.array([2026, 2027, 2028])
trend_forecast  = slope * forecast_years + intercept

# Scenario assumptions (based on KONE management guidance + analyst views)
base_growth = 0.045   # +4.5% — management mid-range guidance
bull_growth = 0.075   # +7.5% — China recovery, modernisation acceleration
bear_growth = 0.005   # +0.5% — prolonged China weakness

base_2026 = sales[-1] * (1 + base_growth)
bull_2026 = sales[-1] * (1 + bull_growth)
bear_2026 = sales[-1] * (1 - 0.005)

# Multi-year compound
base_fc = [sales[-1] * (1+base_growth)**i for i in range(1,4)]
bull_fc = [sales[-1] * (1+bull_growth)**i for i in range(1,4)]
bear_fc = [sales[-1] * (1+bear_growth)**i for i in range(1,4)]

print(f"\n📊 FY2026 Scenarios:")
print(f"   Bull case (+7.5%): €{bull_2026:,.0f}M")
print(f"   Base case (+4.5%): €{base_2026:,.0f}M")
print(f"   Bear case (+0.5%): €{bear_2026:,.0f}M")

In [ ]:
# ── Chart 9: Forecast chart ──
all_years = list(years) + list(forecast_years)

fig = go.Figure()

# Historical actual
fig.add_trace(go.Scatter(
    x=years, y=sales, name='Actual Revenue',
    mode='lines+markers',
    line=dict(color=BLUE, width=3),
    marker=dict(size=8, color=BLUE),
))

# Trend line extended
fig.add_trace(go.Scatter(
    x=all_years, y=list(slope*np.array(all_years)+intercept),
    name='Linear Trend', mode='lines',
    line=dict(color=GRAY, width=1.5, dash='dot'),
))

# Bull scenario
fig.add_trace(go.Scatter(
    x=[2025]+list(forecast_years), y=[sales[-1]]+bull_fc,
    name='Bull Case (+7.5%/yr)', mode='lines+markers',
    line=dict(color=GREEN, width=2, dash='dash'),
    marker=dict(size=6, color=GREEN),
))

# Base scenario
fig.add_trace(go.Scatter(
    x=[2025]+list(forecast_years), y=[sales[-1]]+base_fc,
    name='Base Case (+4.5%/yr)', mode='lines+markers',
    line=dict(color=AMBER, width=2, dash='dash'),
    marker=dict(size=6, color=AMBER),
))

# Bear scenario
fig.add_trace(go.Scatter(
    x=[2025]+list(forecast_years), y=[sales[-1]]+bear_fc,
    name='Bear Case (+0.5%/yr)', mode='lines+markers',
    line=dict(color=RED, width=2, dash='dash'),
    marker=dict(size=6, color=RED),
))

# Shade forecast region
fig.add_vrect(x0=2025.5, x1=2028.5, fillcolor='rgba(26,79,214,0.04)',
              line_width=0, annotation_text='Forecast', annotation_position='top left')

fig.update_layout(
    title='KONE Revenue Forecast — FY2026 to FY2028 (Three Scenarios)',
    yaxis=dict(title='Revenue (MEUR)', tickformat=',', gridcolor='rgba(26,79,214,0.08)'),
    xaxis=dict(title='Year', tickmode='array', tickvals=all_years, gridcolor='rgba(26,79,214,0.08)'),
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
    height=480,
)
fig.show()

In [ ]:
# ── Chart 10: EBIT margin forecast ──
# Historical margins + forward path under each scenario
margin_base = [11.9, 12.2, 12.5]   # Gradual improvement toward strategy target
margin_bull = [12.5, 13.0, 13.5]   # Faster recovery
margin_bear = [11.5, 11.0, 11.0]   # Margin compression

fig = go.Figure()

# Historical
fig.add_trace(go.Scatter(
    x=list(years), y=list(df_income['EBIT_Margin_%']),
    name='Historical EBIT Margin', mode='lines+markers',
    line=dict(color=BLUE, width=3), marker=dict(size=8),
))

for fc_data, label, color, dash in [
    (margin_bull, 'Bull Case', GREEN, 'dash'),
    (margin_base, 'Base Case', AMBER, 'dash'),
    (margin_bear, 'Bear Case', RED, 'dash'),
]:
    fig.add_trace(go.Scatter(
        x=[2025]+list(forecast_years), y=[df_income['EBIT_Margin_%'].iloc[-1]]+fc_data,
        name=label, mode='lines+markers',
        line=dict(color=color, width=2, dash=dash),
        marker=dict(size=6),
    ))

fig.add_hline(y=12.5, line_color=GRAY, line_dash='dot', line_width=1.5,
              annotation_text='Strategy target range', annotation_position='bottom right')

fig.update_layout(
    title='KONE EBIT Margin Forecast — FY2025 to FY2028',
    yaxis=dict(title='EBIT Margin %', ticksuffix='%', range=[8,15], gridcolor='rgba(26,79,214,0.08)'),
    xaxis=dict(tickmode='array', tickvals=all_years, gridcolor='rgba(26,79,214,0.08)'),
    **{k: v for k, v in LAYOUT.items() if k not in ['xaxis','yaxis']},
    height=420,
)
fig.show()

## 6. Summary & Investment Implications

| Metric | FY2023 | FY2024 | FY2025 | Trend |
|--------|--------|--------|--------|-------|
| Revenue (MEUR) | 10,952 | 11,098 | 11,245 | ↑ |
| EBIT Margin | 11.0% | 11.3% | 11.9% | ↑ |
| Adj. EBIT Margin | 11.4% | 11.7% | 12.2% | ↑ |
| Net Income (MEUR) | 932 | 961 | 992 | ↑ |
| Orders Received (MEUR) | 8,578 | 8,759 | 9,087 | ↑ |
| Cash Flow from Ops (MEUR) | 1,485 | 1,589 | 1,761 | ↑↑ |
| ROE % | 33.0% | 33.8% | 34.7% | ↑ |

### Key Findings

**Revenue:** KONE delivered steady €11.2B revenue in FY2025 (+1.3% YoY). The mix shift is significant — Modernisation grew +14.7% YoY to 21% of sales, while New Building Solutions declined -9.1% as Greater China shrank from 23% → 19% of revenue.

**Cost Structure:** Labour remains the largest cost driver (36.4% of sales, +4.7% YoY). Materials costs declined -1.8% YoY, reflecting supply chain normalisation. R&D investment increased to €234M (2.1% of sales), up from €204M — aligned with the 'Rise' strategy focused on digital and energy-efficient solutions.

**Profitability:** EBIT margin improved to 11.9% in FY2025 (from 9.5% trough in FY2022). Adj. EBIT margin reached 12.2%, approaching the long-term strategy target range.

**Cash Generation:** Strong cash flow performance — €1,761M from operations (+10.8% YoY), reflecting KONE's capital-light model and negative working capital advantage.

**Forecast:** Base case projects €11.7B revenue by FY2026 (+4.5%), supported by a resilient service base (>90% retention), modernisation pipeline, and gradual China recovery. Key downside risk remains China property market weakness.

---
*Analysis by Chau Minh Tran | Data source: KONE Annual Review 2025 | Aalto University*